# Poromechanics

Learning goals:
1. Set up and run a 3d simulation with poromechanics
2. Import elliptic fractures from file
3. Use wells to control fluid injection
4. Use line search to stabilize the simulations


In [ ]:
# The usual imports.
import numpy as np
import porepy as pp
from pathlib import Path

from porepy.numerics.nonlinear import line_search
from porepy.applications.boundary_conditions.model_boundary_conditions import(
    HydrostaticBoundaryPressureValues, BoundaryConditionsMechanicsNeumann, LithostaticBoundaryStressValues
)
from porepy.applications.initial_conditions.model_initial_conditions import InitialConditionHydrostaticPressureValues

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

class Geometry:

    def set_domain(self):
        self._domain = pp.Domain({"xmin": -1.0, "xmax": 1.0, "ymin": -1.0, "ymax": 1.0, "zmin": -1.0, "zmax": 1.0})

    def set_fractures(self):
        data = np.genfromtxt(Path('fractures.csv'), delimiter=',', skip_header=1)
        fractures = []
        for row in data:
            # Define an elliptic fracture.
            f = pp.EllipticFracture(row[:3], *row[3:])
            fractures.append(f)
        self._fractures = fractures
        


In [ ]:
class BoundaryConditions(
    HydrostaticBoundaryPressureValues,
    BoundaryConditionsMechanicsNeumann,
    LithostaticBoundaryStressValues,    
):
    pass

In [ ]:
# Set hydrostatic and lithostatic boundary conditions for the flow and mechanics problem
class SimulationSetup(
    Geometry,
    # Activate gravity.
    pp.constitutive_laws.GravityForce,
    # Set the fracture permeability through the cubic law.
    pp.constitutive_laws.CubicLawPermeability,
    #
    BoundaryConditions,
    InitialConditionHydrostaticPressureValues,
    pp.models.solution_strategy.ContactIndicators,
    pp.Poromechanics,
): 
    pass

In [ ]:
# Define time schedule for the simulation.
schedule = np.array([0, pp.HOUR, 10 * pp.HOUR])

# Add initialization time interval.
dt_init = 3 * pp.DAY
schedule += dt_init * 2.5
schedule = np.insert(schedule, 0, 0.0)
# Define injection pressures as list of len = schedule.size. For other protocol
# values, broadcasting of single values is used for simplicity. The following
# schedule is somewhat arbitrary, but meant to represent a ramping up of injection
# pressures over time. The initial low pressure represents a start from near
# hydrostatic conditions.
# We ramp up from 1e5 to 5e6 Pa during initialization (well is closed using a
# Neumann BC), then ramp up to 9e6 Pa at injection start (1 hour), then increase to
# 11e6 Pa after 10 hours, and finally to 15e6 Pa after 200 days.
injection_pressures = [1e5, 5e6, 9e6, 11e6, 15e6]  # [Pa]
# Convenient shortening of simulation schedule for quick simulations. The point is
# that injection_pressures must match the size of schedule.
schedule_length = schedule.size  # Change to 3 or 4 for quicker runs.
schedule = schedule[:schedule_length]
injection_pressures = injection_pressures[:schedule_length]

time_manager = pp.TimeManager(
        schedule=schedule,
        dt_init=dt_init,
        constant_dt=False,
        dt_min_max=(0.1 * pp.MINUTE, max(pp.HOUR, dt_init)),
        iter_optimal_range=(6, 10),  # Allow more iterations than default.
        iter_relax_factors=(0.5, 1.8),  # More aggressive relaxation
    )

In [ ]:
solid_values = pp.solid_values.basalt
solid_values.update(
    {
        "dilation_angle": 0.1,  # [rad]
        # Uncomment next two lines to include elastic fracture deformation, aka
        # "Barton-Bandis" model for normal fracture deformation.
        # "fracture_normal_stiffness": 1.1e8,  # [Pa m^-1]
        # "maximum_elastic_fracture_opening": 1e-3,  # [m]
        "normal_permeability": 1.0e-10,  # [m^2]
        "residual_aperture": 1e-3,  # [m]
        "well_radius": 0.1,  # [m]
    }
)

In [ ]:
model_params = {
    # Set time manager.
    "time_manager": time_manager,
    # Set physical parameters.
    "lithostatic_stress_multipliers": np.array([0.8, 1.2, 1.0]),
    "injection_well_pressures": injection_pressures,
    # The produced fluid is hotter than the injected one.
    "production_well_pressures": pp.ATMOSPHERIC_PRESSURE,  # = 1.01325e5 Pa
    "material_constants": {
        "solid": pp.SolidConstants(**solid_values),  # type: ignore[arg-type]
        "fluid": pp.FluidComponent(**pp.fluid_values.water),  # type: ignore[arg-type]
        "numerical": pp.NumericalConstants(characteristic_displacement=1e-2),
    },
    "reference_variable_values": pp.ReferenceVariableValues(pressure=1e6),
    "units": pp.Units(m=1.0, kg=1.0e5, K=1.0),
    # Set geometry and meshing related parameters.
    "grid_type": "simplex",
    "meshing_arguments": {
        "cell_size": 0.5,
        "cell_size_fracture": 0.3,
        "cell_size_min": 0.1,
    },
    "domain_sizes": 1.0,
    # Line search: Scale the indicator used for the local_line_search (see below)
    # adaptively to increase robustness.
    "adaptive_indicator_scaling": 1,
}

In [ ]:
solver_params = {
    "prepare_simulation": True,
    "nl_max_iterations": 25,  # Max iterations of a nonlinear solver (Newton)
    "nl_convergence_inc_atol": 1e-7,  # Increment norm
    "nl_convergence_res_atol": 1e-7,  # Residual norm
    "nl_divergence_inc_atol": 1e12,
    "nl_divergence_res_atol": 1e12,
    # Line search / Solution Strategies. These are considered "advanced" options,
    # improving the robustness of the nonlinear solver at the cost of some
    # additional computational overhead. Delete/comment the following lines for the
    # default Newton's method.
    "nonlinear_solver": line_search.ConstraintLineSearchNonlinearSolver,
    # Set to 1 to use turn on a residual-based line search. This involves some extra
    # residual evaluations and may be quite costly.
    "global_line_search": 0,
    # Set to 0 to use turn off the tailored line search, see the class
    # ConstraintLineSearchNonlinearSolver. This line search is cheap and has proven
    # effective for (some versions of) this particular simulation setup.
    "local_line_search": 1,
}


In [ ]:
model = SimulationSetup(model_params)
pp.run_time_dependent_model(model, solver_params)


: 

: 